In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    return phonetic

In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩


In [5]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [8]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16681/18310 [04:44<00:16, 98.57it/s] c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [05:02<00:00, 60.49it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,None,ERROR: Could not align 'ไฮโดรเจน' with 'haj˧.d...
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [9]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [11]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18303,ไฮโซ,haj˧.soː˧,1,haj˧.soː˧,None,ERROR: Could not align 'ไฮโซ' with 'haj˧.soː˧'
18304,ไฮโดรคาร์บอน,haj˧.droː˧.kʰaː˧.bɔn˥˩,1,haj˧.droː˧.kʰaː˧.bɔn˦˩,None,ERROR: Could not align 'ไฮโดรคาร์บอน' with 'ha...
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,None,ERROR: Could not align 'ไฮโดรเจน' with 'haj˧.d...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [12]:
errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
2137,ค,kʰɔː˧.kʰwaːj˧,2,kʰɔː˧.kʰwaːj˧,None,ERROR: Could not align 'ค' with 'kʰɔː˧.kʰwaːj˧'
15607,เยื่อตา,jɯa̯˥˩.taː˧,1,jɯə˦˩.taː˧,None,ERROR: Could not align 'เยื่อตา' with 'jɯə˦˩.t...
17285,โดดเดี่ยว,doːt̚˨˩.dia̯w˨˩,1,doːt˨˩.diəw˨˩,None,ERROR: Could not align 'โดดเดี่ยว' with 'doːt˨...
6303,น้ำนิ่งไหลลึก,naːm˦˥.niŋ˥˩.laj˩˩˦.lɯk̚˦˥,1,naːm˦˥.niŋ˦˩.laj˨˥.lɯk˦˥,None,ERROR: Could not align 'น้ำนิ่งไหลลึก' with 'n...
12686,หม้าย,maːj˥˩,1,maːj˦˩,None,ERROR: Could not align 'หม้าย' with 'maːj˦˩'
10560,ลูกโซ่,luːk̚˥˩.soː˥˩,1,luːk˦˩.soː˦˩,None,ERROR: Could not align 'ลูกโซ่' with 'luːk˦˩.s...
3135,ง,ŋɔː˧,2,ŋɔː˧,None,ERROR: Could not align 'ง' with 'ŋɔː˧'
516,กล่าวคือ,klaːw˨˩.kʰɯː˧,1,klaːw˨˩.kʰɯː˧,None,ERROR: Could not align 'กล่าวคือ' with 'klaːw˨...
17502,โมเดล,moː˧.del˧,1,moː˧.del˧,None,ERROR: argument of type 'NoneType' is not iter...
8311,พาฬ,pʰaː˧.la˦˥.,2,pʰaː˧.laʔ˦˥.,None,ERROR: argument of type 'NoneType' is not iter...
